In [1]:
import pandas as pd
import numpy as np 

from sklearn.preprocessing import StandardScaler , MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder , LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split



In [2]:
data = pd.read_csv("C:\\Users\\bingi\\Downloads\\melb_data.csv")
df1 = data.copy()

In [3]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

In [4]:
Q1 = df1['Price'].quantile(0.25)
Q3 = df1['Price'].quantile(0.75)
IQR = Q3 - Q1

outliers = df1[(df1['Price'] < Q1 - 1.5*IQR) | (df1['Price'] > Q3 + 1.5*IQR)]

# Drop those rows
df = df1.drop(outliers.index)



In [5]:
df["Car"].fillna(df["Car"].median(), inplace=True)
df["YearBuilt"].fillna(df["YearBuilt"].median(), inplace=True)
df["CouncilArea"].fillna(df["CouncilArea"].mode()[0], inplace=True)
df.drop("BuildingArea", axis=1, inplace=True) 

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12968 entries, 0 to 13579
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         12968 non-null  object 
 1   Address        12968 non-null  object 
 2   Rooms          12968 non-null  int64  
 3   Type           12968 non-null  object 
 4   Price          12968 non-null  float64
 5   Method         12968 non-null  object 
 6   SellerG        12968 non-null  object 
 7   Date           12968 non-null  object 
 8   Distance       12968 non-null  float64
 9   Postcode       12968 non-null  float64
 10  Bedroom2       12968 non-null  float64
 11  Bathroom       12968 non-null  float64
 12  Car            12968 non-null  float64
 13  Landsize       12968 non-null  float64
 14  YearBuilt      12968 non-null  float64
 15  CouncilArea    12968 non-null  object 
 16  Lattitude      12968 non-null  float64
 17  Longtitude     12968 non-null  float64
 18  Regionname 

In [7]:
# Target
y = df['Price']

X = df.drop(columns = 'Price' , axis = 1)


In [8]:
X['Date'] = pd.to_datetime(X['Date'] , dayfirst=True)

X['Year'] = X['Date'].dt.year
X['Month'] = X['Date'].dt.month
X['Day'] = X['Date'].dt.day

X.drop(columns = 'Date' , inplace = True)

In [9]:
X.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Method', 'SellerG', 'Distance',
       'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'YearBuilt',
       'CouncilArea', 'Lattitude', 'Longtitude', 'Regionname', 'Propertycount',
       'Year', 'Month', 'Day'],
      dtype='object')

In [10]:
X.drop(columns = ['Suburb','Address','Postcode','Lattitude','Longtitude' , 'SellerG','Distance'] , inplace = True)

In [11]:
X['YearBuilt'] = LabelEncoder().fit_transform(X['YearBuilt'])

In [12]:
num_col = X.select_dtypes(include = 'number').columns.to_list()
cat_col = X.select_dtypes(exclude = 'number').columns.to_list()

num_transformation = Pipeline(steps = [('Imputer' , SimpleImputer(strategy='median')),
                                       ('Scaling' , StandardScaler())])

cat_transformation = Pipeline(steps=[('Imputer' , SimpleImputer(strategy='most_frequent')),
                                     ('onehotencoding' , OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers = [('num' , num_transformation , num_col),
                                                 ('cat' , cat_transformation , cat_col)],
                                                 remainder='drop')
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [13]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.20)

In [14]:
print("X_train:", X_train.shape[0])
print("y_train:", y_train.shape[0])
print("X_test:", X_test.shape[0])
print("y_test:", y_test.shape[0])


X_train: 10374
y_train: 10374
X_test: 2594
y_test: 2594


In [15]:
model = LinearRegression()
final_pipeline = Pipeline([('preprocessor' , preprocessor) , ('model' , model)])
final_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [16]:
final_pipeline.fit(X_train , y_train)
y_pred = final_pipeline.predict(X_test)
print('Predictions:',y_pred)

Predictions: [ 290558.3646853  1014570.75901612 1071578.88869413 ...  567926.51409883
  336993.94845601 1679436.6553136 ]


In [17]:
mse = mean_squared_error(y_test , y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test , y_pred)

print('Mean Squared Error:' , mse) 
print('Root Mean Squared Error:' , rmse) 
print('R2 Score:' , r2) 

Mean Squared Error: 75413272861.63281
Root Mean Squared Error: 274614.7717469561
R2 Score: 0.6377260498776525


In [18]:
print("Actual vs Predicted:\n")
for actual, pred in zip(y_test, y_pred):
    print(f"y_actual: {actual} | y_pred: {pred}")


Actual vs Predicted:

y_actual: 386000.0 | y_pred: 290558.36468529515
y_actual: 851500.0 | y_pred: 1014570.759016125
y_actual: 1085000.0 | y_pred: 1071578.8886941322
y_actual: 900000.0 | y_pred: 843615.6566427762
y_actual: 779000.0 | y_pred: 1138953.7966025402
y_actual: 767000.0 | y_pred: 1037227.4488241868
y_actual: 930000.0 | y_pred: 843659.2594948594
y_actual: 870000.0 | y_pred: 969002.2415248884
y_actual: 392500.0 | y_pred: 346317.4337768642
y_actual: 940000.0 | y_pred: 714313.9528963569
y_actual: 1165000.0 | y_pred: 1382888.6827145205
y_actual: 865000.0 | y_pred: 1036804.3601891785
y_actual: 2330000.0 | y_pred: 1546568.748497774
y_actual: 1090000.0 | y_pred: 962446.4008796795
y_actual: 1040000.0 | y_pred: 1077025.2395328593
y_actual: 390000.0 | y_pred: 294055.7550806352
y_actual: 595000.0 | y_pred: 846119.9490592105
y_actual: 1250000.0 | y_pred: 1324428.5739387912
y_actual: 520000.0 | y_pred: 547518.4049152354
y_actual: 808000.0 | y_pred: 882452.2727488636
y_actual: 965000.0 | y_p